In [ ]:
from importlib import reload
import shutil
import sys
import subprocess

import scipy
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../python")
import pymodelica
reload(pymodelica)


# 1. Get fake data

In [ ]:
MODEL_CLASS = "System_2block"
MODEL_EXE_NAME = f"{MODEL_CLASS}_exe"
MODEL_FILE = f"../models/{MODEL_CLASS}.mo"
MODEL_BUILD_DIR = "../build"
MODEL_FILE_LIN = f"{MODEL_EXE_NAME}_linearized_model.py"
MODEL_RESULT = f'{MODEL_BUILD_DIR}/{MODEL_EXE_NAME}_res_init.mat'


OVERRIDE_PARAMS = "../parameters/params_2block_HVeV_NFC.txt"
# OVERRIDE_PARAMS = "../parameters/params_lmo_v1_mod_tes.txt"
EQUILIBRIUM_TIME = 20.0
EQUILIBRIUM_STEP = 2e-3
FINE_STOP_TIME = 0.1
FINE_STEP_SIZE = 3e-6
TOLERANCE = 1e-8
PULSE_ENERGY_EV = 20

CINDEX_TARGET = 1
CINDEX_TARGETGOLD = 2

In [ ]:
run_info = pymodelica.run_model(
    model_exe = MODEL_EXE_NAME,
    build_dir = MODEL_BUILD_DIR,
    override_file = OVERRIDE_PARAMS,
    params={"Edep": PULSE_ENERGY_EV},
    equilibrium_time=EQUILIBRIUM_TIME,
    coarse_step_size=EQUILIBRIUM_STEP,
    fine_stop_time=FINE_STOP_TIME,
    fine_step_size=FINE_STEP_SIZE,
    tolerance=TOLERANCE,
    solver="dassl",
    linearized_model_file=MODEL_FILE_LIN,
    init_result_file=None,
    pulse_result_file=None,
    keep_runs=True,
    extra_coarse_args=None,
    extra_fine_args=None,
)

print(run_info)
config = pymodelica.read_config(run_info["override_file"])

In [ ]:
model = pymodelica.TESModel(f"{MODEL_BUILD_DIR}/{MODEL_FILE_LIN}", 
                 MODEL_FILE, 
                 MODEL_RESULT,
                 config = {},
                 f_min = 1, f_max = 1e6, f_scale="log", points = 10_000)

frequencies, noise_current = model.get_noise(
    Rsh_temperature = 0.2,
    # Rp_temperature = 0.2,
    noise_electronics = 4.2e-12,
    noise_flicker_corner = 20,
    noise_flicker_gamma = 1,
    index_cinput = CINDEX_TARGET)

In [ ]:
time_steps_init, data_init = pymodelica.load(run_info["init_result"])
time_steps, data           = pymodelica.load(run_info["pulse_result"])
print(data.keys())

In [ ]:
# Load simulation result
time_steps_init, data_init = pymodelica.load(run_info["init_result"])

time_steps, data = pymodelica.load(run_info["pulse_result"])
print(data.keys())


# Plot all C and G to check if the system is at equilibrium
y = data_init['c1.T']*1e3
plt.plot(time_steps_init, y, label=f"TES temperature {y[-1]:.3g} mK")
y = data_init['c1.R']*1e3
plt.plot(time_steps_init, y, label=f"TES resistance {y[-1]:.3g} mOhm")
y = data_init['c1.i']*1e6
plt.plot(time_steps_init, y, label=f"TES current {y[-1]:.3g} uA")
y = data_init['c1.i']**2*data_init['c1.R']*1e12
plt.plot(time_steps_init, y, label=f"TES power {y[-1]:.3g} pW")
plt.legend()
plt.xlabel("Time [s]")
plt.grid()
plt.yscale("log")


In [ ]:
data["Rp.R"][-1], data["Rp.R_actual"][-1]

In [ ]:
# Impulse response
t_linear, y_linear_all = model.get_impulse(index_cinput = CINDEX_TARGET, T=np.linspace(0,0.1,100_000))
deposited_energy = config["Edep"] * scipy.constants.e
y_linear = y_linear_all[:,CINDEX_TARGET] * deposited_energy
y_nonlinear = data['c1.i'] - np.mean(data['c1.i'][:10])

collection_linear = model.get_linear_phonon_energy_collection_efficiency(t_linear, y_linear_all, deposited_energy=1)
collection_nonlinear = model.get_nonlinear_phonon_energy_collection_efficiency(time = time_steps, simulation_result = data, deposited_energy=deposited_energy)
plt.plot(t_linear, y_linear, label=f"linear, eff = {100*collection_linear['efficiency']:.2f}%")
plt.plot(time_steps-config["Edep_starttime"], y_nonlinear, label=f"Nonlinear, eff = {100*collection_nonlinear['efficiency']:.2f}%")
plt.legend()
plt.grid()
plt.xlim(-0.0005,0.001)
plt.show()




# Noise
model.plot_noise_current()
plt.show()


# Finite-energy nonlinear template
template_frequencies, dIdE_nonlinear = model.get_nonlinear_energy_template(
    time=time_steps,
    deposited_energy=deposited_energy,
    simulation_result=data,
)
dIdP_linear = model.get_dIdP(index_cinput=CINDEX_TARGET)
plt.loglog(model.frequencies, np.abs(dIdP_linear), label="Linear dI/dP")
plt.loglog(template_frequencies[1:], np.abs(dIdE_nonlinear[1:]), label="Nonlinear pulse dI/dE")
plt.xlabel("Frequency [Hz]")
plt.ylabel("Response [A/W]")
plt.grid(which="both", alpha=0.2)
plt.legend()
plt.show()


# Resolution over the frequency band supported by both responses
resolution_fmin = max(model.frequencies[0], template_frequencies[1])
resolution_fmax = min(model.frequencies[-1], template_frequencies[-1])
dIdP_nonlinear = np.interp(model.frequencies, template_frequencies, np.abs(dIdE_nonlinear))
resolution_linear = model.get_resolution(dIdP_linear, fmin=resolution_fmin, fmax=resolution_fmax, n_interp=1000_000) / collection_linear['efficiency']
resolution_nonlinear = model.get_resolution(dIdP_nonlinear, fmin=resolution_fmin, fmax=resolution_fmax, n_interp=1000_000) / collection_nonlinear['efficiency']

print(f"Resolution band: {resolution_fmin:.3g} Hz to {resolution_fmax:.3g} Hz")
print("Input referred resolution (linear, matched band)[eV]:", resolution_linear)
print("Input referred resolution (nonlinear pulse)[eV]:", resolution_nonlinear)


# Impedance
fig, axs = model.plot_impedance(content="dvdi", fmax=1e5)
# axs[0].set_xlim(right=1e5)
# axs[0].set_ylim(0,0.25)
axs[1].set_aspect('equal', adjustable='box')
plt.tight_layout()



# 2. Smear the truth simulation into fake measurements

The toy likelihood treats bias power, pulse samples, and the real and imaginary impedance components as independent Gaussian measurements.

In [ ]:
rng = np.random.default_rng(20261009)

# Bias-power measurement: one percent uncertainty.
P_true = float(data_init["c1.P_Joule"][-1])
P_sigma = 0.01 * abs(P_true)
P_obs = float(P_true + rng.normal(scale=P_sigma))

# Pulse measurement: retain pre-trigger points and densely sample the fast peak.
pulse_start = config["Edep_starttime"]
pulse_offsets = np.concatenate([
    np.linspace(-30e-6, 0.0, 8, endpoint=False),
    np.geomspace(FINE_STEP_SIZE, 1e-3, 120),
])
t_obs = pulse_start + pulse_offsets
current_truth = data["c1.i"] - np.mean(data["c1.i"][:10])
I_true = np.interp(t_obs, time_steps, current_truth)
I_sigma = np.full_like(I_true, 2e-9)
I_obs = I_true + rng.normal(scale=I_sigma)

# Complex impedance measurement: independent errors on both components.
f_obs = np.logspace(1, 5, 32)
Z_model_grid = model.get_impedance()
Z_true = (
    np.interp(f_obs, model.frequencies, Z_model_grid.real)
    + 1j * np.interp(f_obs, model.frequencies, Z_model_grid.imag)
)
Z_sigma = 0.01 * np.maximum(np.abs(Z_true), 0.02)
Zr_sigma = Z_sigma.copy()
Zi_sigma = Z_sigma.copy()
Z_obs = (
    Z_true.real + rng.normal(scale=Zr_sigma)
    + 1j * (Z_true.imag + rng.normal(scale=Zi_sigma))
)

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].errorbar([0], [P_obs * 1e12], yerr=[P_sigma * 1e12], fmt="o", label="Fake data")
axs[0].axhline(P_true * 1e12, color="C1", label="Truth")
axs[0].set_ylabel("TES bias power [pW]")
axs[0].set_xticks([])
axs[0].legend()
axs[1].plot((t_obs - pulse_start) * 1e6, I_true * 1e6, label="Truth")
axs[1].errorbar((t_obs - pulse_start) * 1e6, I_obs * 1e6, yerr=I_sigma * 1e6, fmt=".", label="Fake data")
axs[1].set_xlabel("Time from pulse [us]")
axs[1].set_ylabel("Current perturbation [uA]")
axs[1].legend()
axs[2].plot(Z_true.real, Z_true.imag, label="Truth")
axs[2].errorbar(Z_obs.real, Z_obs.imag, xerr=Zr_sigma, yerr=Zi_sigma, fmt=".", label="Fake data")
axs[2].set_xlabel(r"Real($Z_{TES}$) [$\Omega$]")
axs[2].set_ylabel(r"Imag($Z_{TES}$) [$\Omega$]")
axs[2].set_aspect("equal", adjustable="box")
axs[2].legend()
for ax in axs:
    ax.grid(alpha=0.2)
plt.tight_layout()

# 3. Recover the toy-model parameters with MCMC

In [ ]:
import scipy.stats as stats

truth_parameters = {
    "TES_alpha": config["TES_alpha"],
    "K1": config["K1"],
}

fit = pymodelica.TESMCMCFit(
    model_exe=MODEL_EXE_NAME,
    build_dir=MODEL_BUILD_DIR,
    full_model_file=MODEL_FILE,
    base_override_file=run_info["override_file"],
    equilibrium_time=EQUILIBRIUM_TIME,
    fine_stop_time=FINE_STOP_TIME,
    coarse_step_size=EQUILIBRIUM_STEP,
    fine_step_size=FINE_STEP_SIZE,
    tolerance=TOLERANCE,
    solver="dassl",
    linearized_model_file="linearized_model.py",
    tes_model_kwargs={"frequencies": f_obs},
    keep_runs=False,
    verbose=False,
)

fit.set_parameters(
    ["TES_alpha", "K1"],
    priors={
        "TES_alpha": stats.uniform(10.0, 70.0),
        "K1": stats.loguniform(truth_parameters["K1"] / 4, truth_parameters["K1"] * 4),
    },
    initial={
        "TES_alpha": 0.8 * truth_parameters["TES_alpha"],
        "K1": 1.25 * truth_parameters["K1"],
    },
    transform={"K1": "log"},
)

fit.set_measurements({
    "bias_power": {
        "value": P_obs,
        "sigma": P_sigma,
        "model_key": "c1.P_Joule",
    },
    "pulse": {
        "time": t_obs,
        "values": I_obs,
        "sigma": I_sigma,
        "model_key": "c1.i",
        "baseline": "subtract_initial",
    },
    "impedance": {
        "frequencies": f_obs,
        "values": Z_obs,
        "sigma_real": Zr_sigma,
        "sigma_imag": Zi_sigma,
        "observable": "impedance",
    },
})

N_WALKERS = 8
N_STEPS = 100
BURN_IN = N_STEPS // 2
np.random.seed(20261009)
sampler = fit.run_mcmc(
    nwalkers=N_WALKERS,
    nsteps=N_STEPS,
    scatter=5e-3,
    progress=True,
)

In [ ]:
theta_chain = sampler.get_chain()
physical_chain = theta_chain.copy()
for parameter_index, parameter_name in enumerate(fit.parameter_names):
    if fit.transforms[parameter_name] == "log":
        physical_chain[:, :, parameter_index] = np.exp(theta_chain[:, :, parameter_index])

flat_samples = physical_chain[BURN_IN:].reshape(-1, len(fit.parameter_names))
print(f"Mean acceptance fraction: {np.mean(sampler.acceptance_fraction):.3f}")
for parameter_index, parameter_name in enumerate(fit.parameter_names):
    lower, median, upper = np.percentile(flat_samples[:, parameter_index], [16, 50, 84])
    print(
        f"{parameter_name}: {median:.6g} +{upper - median:.3g} "
        f"-{median - lower:.3g} (truth {truth_parameters[parameter_name]:.6g})"
    )

fig, axs = plt.subplots(len(fit.parameter_names), 1, figsize=(9, 5), sharex=True)
for parameter_index, (parameter_name, ax) in enumerate(zip(fit.parameter_names, np.atleast_1d(axs))):
    ax.plot(physical_chain[:, :, parameter_index], alpha=0.45)
    ax.axhline(truth_parameters[parameter_name], color="k", linestyle="--", label="Truth")
    ax.set_ylabel(parameter_name)
    ax.legend(loc="upper right")
axs[-1].set_xlabel("MCMC step")
plt.tight_layout()

plt.figure(figsize=(6, 5))
plt.scatter(flat_samples[:, 0], flat_samples[:, 1] * 1e12, s=8, alpha=0.25)
plt.axvline(truth_parameters["TES_alpha"], color="k", linestyle="--")
plt.axhline(truth_parameters["K1"] * 1e12, color="k", linestyle="--")
plt.xlabel(r"$\alpha$")
plt.ylabel(r"$K_1$ [$10^{-12}$ W/K$^n$]")
plt.grid(alpha=0.2)
plt.tight_layout()



import corner
# Flatten the chain to an array of shape (nsamples, ndim)
samples = sampler.get_chain(flat=True)

# Generate the full triangle plot with 1D histograms on the diagonal
fig = corner.corner(
    samples, 
    labels= ["TES_alpha", "K1"]
)

plt.show()

In [ ]:
import corner
physical_chain_flatten = physical_chain[BURN_IN:].reshape(-1, len(fit.parameter_names))
physical_chain_flatten.shape

# Generate the full triangle plot with 1D histograms on the diagonal
fig = corner.corner(
    physical_chain_flatten, 
    labels= ["TES_alpha", "K1"]
)

plt.show()

In [ ]:
flat_theta = sampler.get_chain(discard=BURN_IN, flat=True)
flat_log_probability = sampler.get_log_prob(discard=BURN_IN, flat=True)
best_theta = flat_theta[np.argmax(flat_log_probability)]
best_parameters = fit.theta_to_params(best_theta)
print("Highest-posterior parameters:", best_parameters)

best_result = fit.run_model(best_parameters)
if best_result is None:
    raise RuntimeError("Best-fit simulation failed") from fit.last_error

try:
    _, best_init_data = pymodelica.load(best_result["init_result"])
    best_pulse_time, best_pulse_data = pymodelica.load(best_result["pulse_result"])
    best_current = best_pulse_data["c1.i"] - best_pulse_data["c1.i"][0]
    best_current_obs = np.interp(t_obs, best_pulse_time, best_current)
    best_model = pymodelica.TESModel(
        best_result["linearized_model"],
        MODEL_FILE,
        best_result["init_result"],
        config={},
        frequencies=f_obs,
    )
    Z_best = best_model.get_impedance()
    P_best = float(best_init_data["c1.P_Joule"][-1])

    fig, axs = plt.subplots(1, 3, figsize=(15, 4))
    axs[0].errorbar([0], [P_obs * 1e12], yerr=[P_sigma * 1e12], fmt="o", label="Fake data")
    axs[0].axhline(P_best * 1e12, color="C1", label="Best fit")
    axs[0].set_ylabel("TES bias power [pW]")
    axs[0].set_xticks([])
    axs[0].legend()
    axs[1].errorbar((t_obs - pulse_start) * 1e6, I_obs * 1e6, yerr=I_sigma * 1e6, fmt=".", label="Fake data")
    axs[1].plot((t_obs - pulse_start) * 1e6, best_current_obs * 1e6, label="Best fit")
    # axs[1].set_xscale("symlog", linthresh=5)
    axs[1].set_xlabel("Time from pulse [us]")
    axs[1].set_ylabel("Current perturbation [uA]")
    axs[1].legend()
    axs[2].errorbar(Z_obs.real, Z_obs.imag, xerr=Zr_sigma, yerr=Zi_sigma, fmt=".", label="Fake data")
    axs[2].plot(Z_best.real, Z_best.imag, label="Best fit")
    axs[2].set_xlabel(r"Real($Z_{TES}$) [$\Omega$]")
    axs[2].set_ylabel(r"Imag($Z_{TES}$) [$\Omega$]")
    axs[2].set_aspect("equal", adjustable="box")
    axs[2].legend()
    for ax in axs:
        ax.grid(alpha=0.2)
    plt.tight_layout()
finally:
    if not fit.keep_runs:
        shutil.rmtree(best_result["run_dir"], ignore_errors=True)